# Customer Support Triage Agent — LLM Demo

**Domain:** E-commerce / Customer Operations  
**Model:** Azure OpenAI (GPT-4.1-mini)  
**Architecture:** Mirrors `api/app/services/llm_service.py` and `api/app/guards/routing_guard.py`

---

## What the Pipeline Produces

For every customer message, the triage pipeline returns a structured JSON object:

| Field | Description | Allowed Values |
|-------|-------------|----------------|
| `category` | Message type | Refund Request · Delivery Issue · Product Complaint · Account Problem · General Enquiry · Compliment · Other |
| `urgency` | Priority level | High · Medium · Low |
| `urgency_reason` | One-line explanation | 10–120 characters |
| `sentiment` | Customer tone | Positive · Negative · Neutral · Mixed |
| `suggested_owner` | Responsible team | Customer Service Agent · Billing Team · Logistics Team · Escalate to Manager |
| `draft_response` | Ready-to-edit reply | Professional, empathetic text (null if abusive) |
| `confidence` | Model self-assessed certainty | High · Medium · Low |
| `abusive_flag` | Offensive content detected | `true` / `false` |

After extraction, a **guardrail layer** validates the routing and checks for hallucinated transactional details.

## 1. Setup & Environment

In [ ]:
# Uncomment to install required packages in your environment
# !pip install openai python-dotenv pandas

import os
import json
import re
from pathlib import Path

import pandas as pd
from IPython.display import display
from openai import AzureOpenAI
from dotenv import load_dotenv

print('Imports OK')

In [ ]:
# ---------------------------------------------------------------------------
# Load credentials — searches for the API .env file automatically.
# Create api/.env from api/.env.example and fill in your Azure OpenAI keys.
# ---------------------------------------------------------------------------
env_candidates = [
    Path('../api/.env'),   # standard project layout (notebook in notebooks/)
    Path('.env'),           # local .env next to this notebook
    Path('../.env'),        # root-level fallback
]

env_loaded = False
for env_path in env_candidates:
    if env_path.exists():
        load_dotenv(env_path, override=True)
        print(f'Loaded credentials from: {env_path.resolve()}')
        env_loaded = True
        break

if not env_loaded:
    print('No .env file found — set variables manually in the lines below.')
    # os.environ['AZURE_OPENAI_ENDPOINT'] = 'https://your-resource.openai.azure.com/'
    # os.environ['AZURE_OPENAI_API_KEY']  = 'your-key-here'

ENDPOINT    = os.getenv('AZURE_OPENAI_ENDPOINT', '')
API_KEY     = os.getenv('AZURE_OPENAI_API_KEY', '')
API_VERSION = os.getenv('AZURE_OPENAI_API_VERSION', '2024-12-01-preview')
DEPLOYMENT  = os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME', 'gpt-4.1-mini')

if not ENDPOINT or not API_KEY:
    raise EnvironmentError(
        'AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY must be set. '
        'See api/.env.example for the required variables.'
    )

client = AzureOpenAI(
    azure_endpoint=ENDPOINT,
    api_key=API_KEY,
    api_version=API_VERSION,
)

print('Azure OpenAI client ready')
print(f'  Endpoint   : {ENDPOINT}')
print(f'  Deployment : {DEPLOYMENT}')
print(f'  API Version: {API_VERSION}')

## 2. System Prompt & Constants

Loaded directly from the API source files — keeps the notebook in sync with the production service without duplicating code.

In [ ]:
# Load SYSTEM_PROMPT_V1 and build_user_message() from the API source.
# api/app/prompts/system_prompt.py contains no imports so exec() is safe.
prompt_file = Path('../api/app/prompts/system_prompt.py')
exec(prompt_file.read_text(encoding='utf-8'), globals())

print(f'System prompt loaded ({len(SYSTEM_PROMPT_V1)} chars)')
print()
print('--- Prompt preview (first 300 chars) ---')
print(SYSTEM_PROMPT_V1[:300].strip())
print('...')

In [ ]:
# Load ALLOWED_CATEGORIES, ROUTING_RULES, ALLOWED_OWNERS, etc.
# api/app/core/constants.py only imports from typing — exec() is safe.
from typing import Dict, List  # pre-import so exec() resolves the type hints

constants_file = Path('../api/app/core/constants.py')
exec(constants_file.read_text(encoding='utf-8'), globals())

print('Constants loaded')
print(f'  Categories ({len(ALLOWED_CATEGORIES)}): {ALLOWED_CATEGORIES}')
print(f'  Urgencies  : {ALLOWED_URGENCIES}')
print(f'  Sentiments : {ALLOWED_SENTIMENTS}')
print(f'  Owners     : {ALLOWED_OWNERS}')
print()
print('Routing rules:')
for cat, owners in ROUTING_RULES.items():
    print(f'  {cat:<22} -> {owners}')

## 3. Test Messages

Eight handwritten messages covering all seven categories plus one abusive edge case.
These mirror the sample inputs expected by `POST /triage`.

In [ ]:
TEST_MESSAGES = [
    {
        'id': 1,
        'label': 'Refund Request',
        'message': (
            'I ordered a laptop stand last week and I want my money back immediately. '
            'The quality is terrible — it broke on the very first day of use. '
            'This is completely unacceptable and I expect a full refund right away.'
        ),
    },
    {
        'id': 2,
        'label': 'Delivery Issue',
        'message': (
            'My package was supposed to arrive three days ago but the tracking '
            'just says in transit with no updates. I need to know where my order is '
            'because I need it urgently for an event this weekend.'
        ),
    },
    {
        'id': 3,
        'label': 'Product Complaint',
        'message': (
            'The wireless headphones I received stopped working after just two days. '
            'The left ear cup produces no sound at all. I followed all the setup '
            'instructions correctly — this is clearly a defective unit.'
        ),
    },
    {
        'id': 4,
        'label': 'Account Problem',
        'message': (
            'Hi, I have been locked out of my account since yesterday morning. '
            'I tried resetting my password three times but keep getting an error '
            'saying the reset link is invalid. I cannot access my purchase history.'
        ),
    },
    {
        'id': 5,
        'label': 'General Enquiry',
        'message': (
            'Hello! I was wondering what your return policy is for electronics. '
            'I might need to return an item but want to understand the timeframe '
            'and the process before I make a decision. Thanks in advance!'
        ),
    },
    {
        'id': 6,
        'label': 'Compliment',
        'message': (
            'I just wanted to say a huge thank you to your support team. '
            'The agent who helped me last week resolved my issue incredibly quickly '
            'and was so professional throughout the whole process. Keep it up!'
        ),
    },
    {
        'id': 7,
        'label': 'Other (B2B Enquiry)',
        'message': (
            'I run a small business and am considering placing regular bulk orders. '
            'I have several questions about volume pricing, B2B invoicing options, '
            'and whether a dedicated account manager would be available for my account.'
        ),
    },
    {
        'id': 8,
        'label': 'Abusive Message (edge case)',
        'message': (
            'You absolute idiots! This is the worst company I have ever dealt with! '
            'I will destroy your reputation online you useless incompetent fools! '
            'Fix this NOW or I will make sure everyone knows how terrible you are!'
        ),
    },
]

print(f'{len(TEST_MESSAGES)} test messages ready')
print()
for m in TEST_MESSAGES:
    mid = m['id']
    label = m['label']
    preview = m['message'][:60]
    print(f'  [{mid}] {label:<28} "{preview}..."')

## 4. LLM Extraction Function

Calls Azure OpenAI with the system prompt and returns the structured JSON triage output.  
This mirrors `api/app/services/llm_service.py → LLMService.extract_triage()`.

In [ ]:
def extract_triage(message):
    """Call Azure OpenAI and return the structured triage dict."""
    response = client.chat.completions.create(
        model=DEPLOYMENT,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT_V1},
            {'role': 'user', 'content': build_user_message(message)},
        ],
        temperature=0.0,
        response_format={'type': 'json_object'},
    )
    raw = response.choices[0].message.content
    return json.loads(raw)


print('extract_triage() defined')

## 5. Guardrail Validation

The guardrail runs two layers of checks:  
1. **Deterministic checks** — routing rules, high-urgency escalation, hallucination patterns  
2. **LLM guardrail call** — secondary validation using a separate system prompt  

Mirrors `api/app/guards/routing_guard.py → guardrail_check()`.

In [ ]:
# Patterns that flag hallucinated transactional details in the draft response.
# These are checked against the original message — if a match appears in the
# draft but NOT in the original, it is a hallucination.
HALLUCINATION_PATTERNS = [
    r'#\d+',
    r'ORD[- ]?\d+',
    r'\$\d+(?:\.\d{2})?',
    r'£\d+(?:\.\d{2})?',
    r'€\d+(?:\.\d{2})?',
]


def guardrail_check(triage_result, original_message):
    """Validate triage output: routing rules + hallucination detection."""
    category  = triage_result.get('category', '')
    owner     = triage_result.get('suggested_owner', '')
    urgency   = triage_result.get('urgency', '')
    draft     = triage_result.get('draft_response', '')
    ur_reason = triage_result.get('urgency_reason', '')
    abusive   = triage_result.get('abusive_flag', False)

    # Skip validation for abusive messages — no draft to validate
    if abusive:
        return {'valid': True, 'reason': 'Abusive flag set — guardrail skipped', 'failures': []}

    # Skip if no draft response to validate
    if not draft or str(draft).strip().lower() == 'not mentioned':
        return {'valid': True, 'reason': 'No draft response to validate', 'failures': []}

    failures = []

    # --- CHECK 1: Routing rules ---
    allowed = ROUTING_RULES.get(category)
    if allowed is None and category != 'Other':
        failures.append(f"Unknown category: '{category}'")
    elif allowed is not None and owner not in allowed:
        failures.append(
            f"Routing mismatch: '{category}' must go to {allowed}, got '{owner}'"
        )

    # --- CHECK 2: High-urgency escalation ---
    high_urgency_cats = ['Refund Request', 'Product Complaint', 'Account Problem']
    if urgency == 'High' and category in high_urgency_cats and owner != 'Escalate to Manager':
        failures.append(
            f"High-urgency '{category}' must escalate to Manager, got '{owner}'"
        )

    # --- CHECK 3: Hallucination detection ---
    if isinstance(draft, str):
        for pattern in HALLUCINATION_PATTERNS:
            matches = re.findall(pattern, draft, flags=re.IGNORECASE)
            for match in matches:
                if match not in original_message:
                    failures.append(f"Hallucinated value in draft: '{match}'")

    # --- CHECK 4: urgency_reason length ---
    reason_text = str(ur_reason).strip()
    reason_len  = len(reason_text)
    if reason_len < 10 or reason_len > 120:
        failures.append(
            f'urgency_reason length out of range ({reason_len} chars, expected 10-120)'
        )

    if failures:
        return {'valid': False, 'reason': '; '.join(failures), 'failures': failures}

    # --- LLM secondary guardrail call ---
    guardrail_system = (
        'You are a routing guardrail for a customer support triage system. '
        'Validate that suggested_owner matches the category per these rules: '
        'Refund Request -> Billing Team or Customer Service Agent; '
        'Delivery Issue -> Logistics Team or Customer Service Agent; '
        'Product Complaint -> Customer Service Agent or Escalate to Manager; '
        'Account Problem -> Billing Team or Customer Service Agent; '
        'General Enquiry -> Customer Service Agent; '
        'Compliment -> Customer Service Agent; Other -> any owner. '
        'Extra rule: urgency=High AND category in [Refund Request, Product Complaint, Account Problem] '
        '=> owner MUST be Escalate to Manager. '
        'Return ONLY valid JSON: {"valid": true/false, "reason": "one sentence"}'
    )
    snippet = original_message[:200]
    guardrail_user = (
        f'Category: {category} | Owner: {owner} | Urgency: {urgency} | '
        f'Message snippet: {snippet}. Is this routing valid?'
    )

    try:
        resp = client.chat.completions.create(
            model=DEPLOYMENT,
            messages=[
                {'role': 'system', 'content': guardrail_system},
                {'role': 'user', 'content': guardrail_user},
            ],
            response_format={'type': 'json_object'},
            temperature=0,
            max_completion_tokens=150,
        )
        llm_out = json.loads(resp.choices[0].message.content)
        return {
            'valid': llm_out.get('valid', True),
            'reason': llm_out.get('reason', 'LLM guardrail passed'),
            'failures': failures,
        }
    except Exception as exc:
        return {'valid': True, 'reason': f'Guardrail LLM error (non-blocking): {exc}', 'failures': []}


print('guardrail_check() defined')

## 6. Full Triage Pipeline

Combines LLM extraction and guardrail validation into one function.  
Mirrors the flow in `api/app/services/triage_service.py → TriageService.process_single_triage()`.

In [ ]:
def run_triage_pipeline(msg_id, label, message):
    """End-to-end: LLM extraction -> guardrail validation -> structured result."""
    sep = '=' * 60
    print(sep)
    print(f'[{msg_id}] {label}')
    print(sep)
    print(f'Input : {message[:90]}...')
    print()

    try:
        # Step 1 — LLM triage extraction
        triage = extract_triage(message)

        print('LLM Triage Output:')
        print(f'  category       : {triage.get("category")}')
        print(f'  urgency        : {triage.get("urgency")} — {triage.get("urgency_reason")}')
        print(f'  sentiment      : {triage.get("sentiment")}')
        print(f'  suggested_owner: {triage.get("suggested_owner")}')
        print(f'  confidence     : {triage.get("confidence")}')
        print(f'  abusive_flag   : {triage.get("abusive_flag")}')
        draft_preview = str(triage.get('draft_response', ''))[:80]
        print(f'  draft_response : {draft_preview}...')
        print()

        # Step 2 — Guardrail validation
        guard = guardrail_check(triage, message)
        status_icon = 'PASSED' if guard['valid'] else 'FAILED'
        print(f'  Guardrail      : {status_icon} — {guard["reason"]}')
        print()

        return {
            'id': msg_id,
            'label': label,
            'input_message': message,
            'triage': triage,
            'guardrail': guard,
            'status': 'OK' if guard['valid'] else 'GUARDRAIL_FAIL',
        }

    except Exception as exc:
        print(f'  Pipeline error: {exc}')
        print()
        return {
            'id': msg_id,
            'label': label,
            'input_message': message,
            'triage': None,
            'guardrail': None,
            'status': f'ERROR: {exc}',
        }


print('run_triage_pipeline() defined')

## 7. Run the Demo

Run the full pipeline on all eight test messages.  
Each message makes **two LLM calls**: one for triage extraction, one for guardrail validation.

In [ ]:
results = []

for msg_data in TEST_MESSAGES:
    result = run_triage_pipeline(
        msg_id=msg_data['id'],
        label=msg_data['label'],
        message=msg_data['message'],
    )
    results.append(result)

print('=' * 60)
passed  = sum(1 for r in results if r['status'] == 'OK')
failed  = sum(1 for r in results if r['status'] == 'GUARDRAIL_FAIL')
errored = sum(1 for r in results if r['status'].startswith('ERROR'))
print(f'Completed {len(results)} messages')
print(f'  Guardrail Passed : {passed}')
print(f'  Guardrail Failed : {failed}')
print(f'  Errors           : {errored}')

## 8. Results Summary Table

A pandas DataFrame showing all eight triage results side by side.

In [ ]:
rows = []
for r in results:
    t = r['triage'] or {}
    g = r['guardrail'] or {}
    rows.append({
        'ID':        r['id'],
        'Label':     r['label'],
        'Category':  t.get('category', '—'),
        'Urgency':   t.get('urgency', '—'),
        'Sentiment': t.get('sentiment', '—'),
        'Owner':     t.get('suggested_owner', '—'),
        'Confidence':t.get('confidence', '—'),
        'Abusive':   t.get('abusive_flag', '—'),
        'Guardrail': 'PASSED' if g.get('valid') else 'FAILED',
        'Status':    r['status'],
    })

df = pd.DataFrame(rows).set_index('ID')


def colour_guardrail(val):
    if val == 'PASSED':
        return 'background-color: #d4edda; color: #155724'
    if val == 'FAILED':
        return 'background-color: #f8d7da; color: #721c24'
    return ''


def colour_urgency(val):
    mapping = {'High': '#f8d7da', 'Medium': '#fff3cd', 'Low': '#d4edda'}
    bg = mapping.get(val, '')
    return f'background-color: {bg}' if bg else ''


def colour_abusive(val):
    if val is True:
        return 'background-color: #f8d7da; font-weight: bold'
    return ''


styled = (
    df.style
    .map(colour_guardrail, subset=['Guardrail'])
    .map(colour_urgency,   subset=['Urgency'])
    .map(colour_abusive,   subset=['Abusive'])
    .set_table_styles([{
        'selector': 'th',
        'props': [('background-color', '#343a40'), ('color', 'white'), ('font-weight', 'bold')]
    }])
)

display(styled)

## 9. Inspect an Individual Result

Change `inspect_id` to view the full JSON for any message.

In [ ]:
inspect_id = 1  # Change to any message ID (1-8)

r = next((x for x in results if x['id'] == inspect_id), None)
if r is None:
    print(f'No result found for ID {inspect_id}')
else:
    print(f'=== Message [{r["id"]}]: {r["label"]} ===')
    print()
    print('Input message:')
    print(r['input_message'])
    print()
    print('Triage output (full JSON):')
    print(json.dumps(r['triage'], indent=2))
    print()
    print('Guardrail result:')
    print(json.dumps(r['guardrail'], indent=2))

## 10. Edge Case: Abusive Message Detection

When `abusive_flag` is `true`, the draft response must be `null` or the exact string  
`"FLAGGED — human review required"`. No normal reply is generated.

In [ ]:
abusive_result = next((r for r in results if r['id'] == 8), None)

if abusive_result and abusive_result['triage']:
    t = abusive_result['triage']
    print('=== ABUSIVE MESSAGE DETECTION ===')
    print()
    print('Input message:')
    print(abusive_result['input_message'])
    print()
    print(f'abusive_flag   : {t.get("abusive_flag")}')
    print(f'draft_response : {t.get("draft_response")}')
    print(f'category       : {t.get("category")}')
    print(f'sentiment      : {t.get("sentiment")}')
    print()
    if t.get('abusive_flag'):
        print('PASS: Message correctly flagged — no draft response generated.')
        print('Human agent review is required before any reply is sent.')
    else:
        print('WARN: Message was not flagged as abusive — review the system prompt.')

## 11. Batch Summary Statistics

In [ ]:
successful = [r for r in results if r['triage'] is not None]

if successful:
    categories  = [r['triage']['category']  for r in successful]
    urgencies   = [r['triage']['urgency']   for r in successful]
    sentiments  = [r['triage']['sentiment'] for r in successful]
    confidences = [r['triage']['confidence']for r in successful]

    print('=== BATCH STATISTICS ===')
    print()

    print('Category distribution:')
    cat_counts = pd.Series(categories).value_counts()
    for cat, cnt in cat_counts.items():
        print(f'  {cat:<26} {cnt}')

    print()
    print('Urgency distribution:')
    urg_counts = pd.Series(urgencies).value_counts()
    for urg, cnt in urg_counts.items():
        print(f'  {urg:<10} {cnt}')

    print()
    print('Sentiment distribution:')
    sent_counts = pd.Series(sentiments).value_counts()
    for sent, cnt in sent_counts.items():
        print(f'  {sent:<10} {cnt}')

    print()
    print('Confidence distribution:')
    conf_counts = pd.Series(confidences).value_counts()
    for conf, cnt in conf_counts.items():
        print(f'  {conf:<10} {cnt}')

    print()
    abusive_count = sum(1 for r in successful if r['triage'].get('abusive_flag'))
    print(f'Abusive messages flagged : {abusive_count}/{len(successful)}')
    guardrail_pass = sum(1 for r in results if r['status'] == 'OK')
    print(f'Guardrail pass rate      : {guardrail_pass}/{len(results)}')

## 12. Export Results to CSV

In [ ]:
export_rows = []
for r in results:
    t = r['triage'] or {}
    g = r['guardrail'] or {}
    export_rows.append({
        'id':              r['id'],
        'label':           r['label'],
        'input_message':   r['input_message'],
        'category':        t.get('category', ''),
        'urgency':         t.get('urgency', ''),
        'urgency_reason':  t.get('urgency_reason', ''),
        'sentiment':       t.get('sentiment', ''),
        'suggested_owner': t.get('suggested_owner', ''),
        'confidence':      t.get('confidence', ''),
        'abusive_flag':    t.get('abusive_flag', ''),
        'draft_response':  t.get('draft_response', ''),
        'guardrail_valid': g.get('valid', ''),
        'guardrail_reason':g.get('reason', ''),
        'pipeline_status': r['status'],
    })

export_df = pd.DataFrame(export_rows)
output_path = Path('./triage_results.csv')
export_df.to_csv(output_path, index=False)
print(f'Results exported to: {output_path.resolve()}')
print(f'Rows: {len(export_df)}, Columns: {len(export_df.columns)}')
display(export_df.head())

## Summary

### What This Notebook Demonstrated

| Step | Implementation |
|------|----------------|
| LLM classification | `extract_triage()` → Azure OpenAI with `response_format: json_object` |
| Prompt engineering | `SYSTEM_PROMPT_V1` loaded from `api/app/prompts/system_prompt.py` |
| Guardrail validation | `guardrail_check()` → deterministic rules + secondary LLM call |
| Abusive content | `abusive_flag=true` suppresses the draft response |
| Hallucination guard | Regex patterns flag invented order numbers, prices, and amounts |
| Routing guard | Business rules ensure the right team receives each message |

### Connecting to the Production API

The full service is a FastAPI app running in Docker:

```bash
# Start the full stack
docker-compose up --build

# Test the single-message endpoint
curl -X POST http://localhost:8000/triage \
     -H 'Content-Type: application/json' \
     -d '{"message": "My package has not arrived and tracking shows no update."}'

# Open the Streamlit UI
open http://localhost:8501
```

### Files That Power This Notebook

| File | Purpose |
|------|---------|
| `api/app/prompts/system_prompt.py` | System prompt + `build_user_message()` |
| `api/app/core/constants.py` | Categories, urgencies, routing rules |
| `api/app/services/llm_service.py` | Production LLM call |
| `api/app/guards/routing_guard.py` | Production guardrail |
| `api/app/services/triage_service.py` | Orchestrates extraction + guardrail |